In [87]:
# ============================================================
# ECOSTRESS NIGHTTIME LST PROCESSING PIPELINE
# - Parses APPEEARS ECOSTRESS filenames
# - Applies QC filtering
# - Keeps nighttime overpasses only
# - Aggregates to INSEE 200 m grid
# - Computes Nighttime Thermal Inertia (NTI)
# - Uses Lambert-93 (EPSG:2154)
# ============================================================

import geopandas as gpd
import rioxarray as rxr
import rasterio
from rasterio.features import rasterize
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from tqdm import tqdm


In [88]:
# Define paths and parameters
DATA_ROOT = Path("C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/data")
GRID_PATH = "C:/Users/kbonsu/Desktop/AICC Conference/idf_200_grid.shp"
IDF_PATH = "C:/Users/kbonsu/Desktop/AICC Conference/ile-de-france.shp"

GRID_ID_FIELD = "idcar_200m"     # FINAL spatial identifier

OUT_LST = "C:/Users/kbonsu/Desktop/AICC Conference/ecostress_night_LST_200m.csv"
OUT_NTI = "C:/Users/kbonsu/Desktop/AICC Conference/ecostress_heat_retention_200m.csv"
OUT_SUMMER_NTI = "C:/Users/kbonsu/Desktop/AICC Conference/ecostress_summer_heat_retention_200m.csv"

In [89]:
# Ile-de-France boundary 
idf = gpd.read_file(IDF_PATH).to_crs("EPSG:2154")

# INSEE 200 m grid (analysis units)
grid = gpd.read_file(GRID_PATH).to_crs("EPSG:2154")
grid = grid[[GRID_ID_FIELD, "geometry"]].reset_index(drop=True)

# Temporary numeric ID for rasterization only
grid["tmp_id"] = grid.index.astype("int32")
grid_lookup = grid[["tmp_id", GRID_ID_FIELD]].copy()


In [90]:
# Function to parse ECOSTRESS datetime from filename (e.g ECO_L2_LSTE.002_LST_doy2018224162117_aid0001)
# Converts UTC → Europe/Paris local time
def parse_ecostress_datetime(fname):
    part = fname.split("_doy")[1].split("_")[0]
    year = int(part[:4])
    doy  = int(part[4:7])
    time = part[7:13]

    dt_utc = datetime.strptime(
        f"{year} {doy} {time}",
        "%Y %j %H%M%S"
    )

    dt_utc = dt_utc.replace(tzinfo=ZoneInfo("UTC"))
    dt_local = dt_utc.astimezone(ZoneInfo("Europe/Paris"))
    
    return dt_local


In [91]:
# Function to load and preprocess ECOSTRESS LST, cloud mask, and QC layers

def load_ecostress_lst(lst_path, cloud_path, qc_path, idf):

    lst = rxr.open_rasterio(lst_path, masked=True).squeeze()
    cloud = rxr.open_rasterio(cloud_path, masked=True).squeeze()
    qc = rxr.open_rasterio(qc_path, masked=True).squeeze()

    # 1. Clear-sky only (mandatory)
    lst = lst.where(cloud == 0)

    # 2. Remove only invalid QC (do NOT over-filter)
    lst = lst.where(qc != 255)

    # 3. Reproject to Lambert-93
    lst = lst.rio.reproject("EPSG:2154")

    # 4. Clip to EXACT download extent (Île-de-France)
    lst = lst.rio.clip(
        idf.geometry,
        idf.crs,
        drop=True
    )

    # 5. ECOSTRESS L2 LSTE v002 scaling
    # LST (K) = DN × 0.02 → °C
    lst = (lst * 0.02) - 273.15

    # Soft sanity check (warn only)
    vmin = float(lst.min())
    vmax = float(lst.max())
    if vmin < -20 or vmax > 60:
        print(f"⚠️ Warning: wide LST range ({vmin:.1f}, {vmax:.1f})")

    return lst


In [92]:
# Function to rasterize INSEE grid to match ECOSTRESS LST raster (for zonal stats)

def rasterize_grid(grid, ref_raster):
    shapes = (
        (geom, val)
        for geom, val in zip(grid.geometry, grid["tmp_id"])
    )

    return rasterize(
        shapes=shapes,
        out_shape=ref_raster.shape,
        transform=ref_raster.rio.transform(),
        fill=-1,
        dtype="int32"
    )


In [93]:
# Function to compute zonal mean LST for each grid cell 
def zonal_mean_fast(lst, grid_raster):
    data = lst.values
    zones = grid_raster

    valid = (zones >= 0) & (~np.isnan(data))

    df = pd.DataFrame({
        "tmp_id": zones[valid],
        "LST": data[valid]
    })

    return df.groupby("tmp_id")["LST"].mean()


In [94]:
# Main processing loop
records = []

for folder in sorted(DATA_ROOT.iterdir()):
    if not folder.is_dir():
        continue

    print(f"\nProcessing folder: {folder.name}")
    lst_files = list(folder.glob("*_LST_doy*.tif"))

    for lst_file in tqdm(lst_files, desc=folder.name):

        cloud_file = lst_file.with_name(
            lst_file.name.replace("_LST_", "_cloud_mask_")
        )
        qc_file = lst_file.with_name(
            lst_file.name.replace("_LST_", "_QC_")
        )

        if not cloud_file.exists() or not qc_file.exists():
            continue

        dt = parse_ecostress_datetime(lst_file.name)

        # Nighttime filter (ECOSTRESS reality)
        if not (dt.hour >= 22 or dt.hour <= 5):
            continue

        lst = load_ecostress_lst(
            lst_file,
            cloud_file,
            qc_file,
            idf
        )

        grid_raster = rasterize_grid(grid, lst)
        means = zonal_mean_fast(lst, grid_raster)

        df = means.reset_index()
        df["datetime"] = dt
        df["date"] = dt.date()
        df["hour"] = dt.hour
        df["year"] = dt.year
        df["month"] = dt.month

        records.append(df)



Processing folder: ECOSTRESS_2018-2019


ECOSTRESS_2018-2019:  81%|████████  | 329/408 [04:21<02:25,  1.84s/it]

⚠️ Warning: wide LST range (-131.2, 74.2)


ECOSTRESS_2018-2019: 100%|██████████| 408/408 [05:36<00:00,  1.21it/s]



Processing folder: ECOSTRESS_2020-2021


...

Olympic Site: 0it [00:00, ?it/s]



Processing folder: summer_ndvi_ndbi_dem


summer_ndvi_ndbi_dem: 0it [00:00, ?it/s]



Processing folder: summer_ndvi_ndbi_dem_2154


summer_ndvi_ndbi_dem_2154: 0it [00:00, ?it/s]


In [99]:
# Merge all records and join back to original grid IDs
ecostress_df = pd.concat(records, ignore_index=True)

ecostress_df = (
    ecostress_df
    .merge(grid_lookup, on="tmp_id", how="left")
    .drop(columns=["tmp_id"])
)



In [109]:
ecostress_df.head() 

,LST,datetime,date,hour,year,month,idcar_200m
0,21.750000,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2805800E3777400
1,22.575005,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2805800E3777600
2,22.654999,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2805800E3777800
3,21.812504,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2806000E3776200
4,22.779236,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2806000E3777600


In [100]:
ecostress_df.describe()

,LST,hour,year,month
count,1.206942e+07,1.206942e+07,1.206942e+07,1.206942e+07
mean,1.157333e+01,7.272657e+00,2.021604e+03,7.179442e+00
std,6.490305e+00,8.559279e+00,2.126969e+00,2.812938e+00
min,-7.533000e+01,0.000000e+00,2.018000e+03,1.000000e+00
25%,6.562500e+00,2.000000e+00,2.020000e+03,5.000000e+00
50%,1.238733e+01,3.000000e+00,2.022000e+03,7.000000e+00
75%,1.625667e+01,5.000000e+00,2.023000e+03,9.000000e+00
max,3.032700e+02,2.300000e+01,2.025000e+03,1.200000e+01


In [101]:
ecostress_df["LST"].quantile([0.001, 0.01, 0.99, 0.999])

0.001    -4.198569
0.010    -1.603333
0.990    24.342308
0.999    33.056493
Name: LST, dtype: float64

In [102]:
ecostress_df["month"] = ecostress_df["datetime"].dt.month

ecostress_df.groupby("month")["LST"].mean()


month
1      4.091865
2      1.478680
3      2.580386
4     10.105347
5     10.688075
6     17.327547
7     18.327007
8     17.402897
9     14.113669
10     9.899505
11     6.141269
12     2.832694
Name: LST, dtype: float32

In [108]:
# Compute mean LST for each calendar month (across all years) and save results
monthly_mean =  ecostress_df.groupby('month')['LST'].agg(['mean', 'std', 'count']).round(2)

print("Mean LST per month (across all years):")

print(monthly_mean)


monthly_mean.to_csv(
    'C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/ecostress_monthly_mean_LST.csv'
)


Mean LST per month (across all years):
        mean   std    count
month                      
1       4.09  2.15   374265
2       1.48  3.15   329532
3       2.58  2.76  1380077
4      10.11  2.01   105620
5      10.69  2.96  1252755
6      17.33  4.65   372049
7      18.33  3.39  2726903
8      17.40  5.03   207412
9      14.11  2.72  3411091
10      9.90  2.06   203970
11      6.14  2.70  1396500
12      2.83  2.55   309246


In [110]:
ecostress_df.columns

Index(['LST', 'datetime', 'date', 'hour', 'year', 'month', 'idcar_200m'], dtype='str')

In [130]:
# Bin nighttime hours into "late night" (22, 23, 0, 1) vs "early morning" (3, 4, 5)
# 2 am was left out because it's a transitional hour that can belong to either bin depending on the overpass timing

ecostress_df = ecostress_df.copy()

#Assign night bins
ecostress_df["night_bin"] = pd.NA

# Late night: 22, 23, 0, 1
ecostress_df.loc[
    (ecostress_df["hour"] >= 22) | (ecostress_df["hour"] <= 1),
    "night_bin"
] = "late_night"

# Early morning: 3, 4, 5
ecostress_df.loc[
    (ecostress_df["hour"] >= 3) & (ecostress_df["hour"] <= 5),
    "night_bin"
] = "early_morning"

# Drop transitional hour (2am)
ecostress_df = ecostress_df.dropna(subset=["night_bin"])

#Aggregate using median + count
grouped = (
    ecostress_df
    .groupby([GRID_ID_FIELD, "year", "month", "night_bin"])["LST"]
    .agg(["median", "count"])
    .reset_index()
)

#Require minimum number of observations per bin
MIN_OBS = 2
grouped = grouped[grouped["count"] >= MIN_OBS]

# Pivot to wide format
nti = (
    grouped
    .pivot(index=[GRID_ID_FIELD, "year", "month"],
           columns="night_bin",
           values="median")
    .reset_index()
)

# Require both bins to exist
nti = nti.dropna(
    subset=["late_night", "early_morning"]
)

# Compute Nighttime Thermal Inertia (NTI) = late night LST - early morning LST
nti["NTI"] = nti["late_night"] - nti["early_morning"]

# Winsorize extreme NTI values
q1 = nti["NTI"].quantile(0.01)
q99 = nti["NTI"].quantile(0.99)

nti["NTI"] = nti["NTI"].clip(q1, q99)



In [131]:
nti.head()

night_bin,idcar_200m,year,month,early_morning,late_night,NTI
7,CRS3035RES200mN2805800E3777600,2020,9,9.602505,15.939995,6.337490
9,CRS3035RES200mN2805800E3777600,2020,11,3.952499,6.795001,2.842502
10,CRS3035RES200mN2805800E3777600,2021,3,1.696676,2.445000,0.748324
12,CRS3035RES200mN2805800E3777600,2022,11,7.320004,4.885002,-2.435001
17,CRS3035RES200mN2805800E3777800,2018,9,14.645828,14.384995,-0.260834


In [132]:
nti.groupby('month')['NTI'].agg(['mean', 'std', 'count']).round(2)

,mean,std,count
month,,,
1,-1.67,1.42,8300
3,2.60,1.80,82741
5,3.16,1.04,42410
7,3.45,1.73,204445
9,1.61,2.06,272541
11,-1.00,2.60,127910


In [134]:
# Mean NTI by year for July (month == 7)
july_nti = nti[nti['month'] == 7] if 'month' in nti.columns else nti

mean_nti_by_year = july_nti.groupby('year')['NTI'].agg(['mean', 'std', 'count']).round(3)

print("Mean NTI by year (July only):")
print(mean_nti_by_year)

Mean NTI by year (July only):
       mean    std  count
year                     
2019  2.994  1.412  60730
2020  3.292  1.486   1510
2021  6.688  0.855   7675
2022  4.825  1.046  48205
2023  2.654  1.444  43987
2025  2.799  1.603  42338


In [135]:
# nti currently filtered to July and MIN_OBS = 2

# Keep July only
nti_july = nti[nti["month"] == 7].copy()

# Count distinct years per grid cell
years_per_grid = (
    nti_july
    .groupby(GRID_ID_FIELD)["year"]
    .nunique()
    .reset_index(name="n_years")
)

# Summary distribution
years_per_grid["n_years"].value_counts().sort_index()


n_years
1    11799
2    14235
3    10297
4    23674
5     7467
6      209
Name: count, dtype: int64

In [136]:
nti_july.shape[0] / grid.shape[0]

2.194958289941273

In [137]:
overall_var = nti_july["NTI"].var()
overall_mean = nti_july["NTI"].mean()

grid_means = (
    nti_july
    .groupby(GRID_ID_FIELD)["NTI"]
    .mean()
)

between_grid_var = grid_means.var()

year_means = (
    nti_july
    .groupby("year")["NTI"]
    .mean()
)

between_year_var = year_means.var()

within_grid_var = (
    nti_july
    .groupby(GRID_ID_FIELD)["NTI"]
    .var()
    .mean()
)

print("Overall variance:", overall_var)
print("Between-grid variance:", between_grid_var)
print("Between-year variance:", between_year_var)
print("Avg within-grid variance:", within_grid_var)



Overall variance: 2.9996808628059974
Between-grid variance: 0.9284248748695527
Between-year variance: 2.5155711806952668
Avg within-grid variance: 3.158350496911432


In [138]:
nti_july.head()

night_bin,idcar_200m,year,month,early_morning,late_night,NTI
185,CRS3035RES200mN2806400E3787000,2019,7,14.997168,19.900002,4.902834
299,CRS3035RES200mN2806600E3786000,2019,7,14.903335,16.898895,1.995561
398,CRS3035RES200mN2806800E3786400,2019,7,13.869987,16.498011,2.628023
414,CRS3035RES200mN2806800E3786600,2019,7,14.529993,17.160858,2.630865
524,CRS3035RES200mN2807000E3786200,2019,7,14.838003,16.486668,1.648664


In [140]:
nti_july.describe()

night_bin,year,month,early_morning,late_night,NTI
count,204445.000000,204445.0,204445.000000,204445.000000,204445.000000
mean,2021.892959,7.0,16.156736,19.662588,3.453182
std,2.192721,0.0,1.766720,1.913755,1.731959
min,2019.000000,7.0,9.434003,1.733337,-4.280369
25%,2019.000000,7.0,14.800379,18.344641,2.172328
50%,2022.000000,7.0,15.953331,19.457085,3.362883
75%,2023.000000,7.0,17.347651,20.820770,4.692575
max,2025.000000,7.0,23.301941,46.780769,7.197543


In [ ]:
# Add geometry to summer_nti and save as GeoPackage
nti_july = nti_july.merge(
    grid[[GRID_ID_FIELD, 'geometry']],
    left_on=GRID_ID_FIELD,
    right_on=GRID_ID_FIELD,
    how='left'
)

nti_july = gpd.GeoDataFrame(nti_july, geometry='geometry', crs=grid.crs)


✅ Summer NTI saved as GeoPackage with geometry


In [150]:
nti_july.columns

Index(['idcar_200m', 'year', 'month', 'early_morning', 'late_night', 'NTI',
       'geometry'],
      dtype='str')

In [147]:
paris = gpd.read_file(r"C:\Users\kbonsu\Desktop\AICC Conference\ECOSTRESS_data_processing\paris_coeur_hypercenter.gpkg").to_crs(grid.crs)
paris_boundary = paris.dissolve()
nti_july_paris = gpd.overlay(
    nti_july, 
    paris_boundary,
    how = "intersection"
    )


In [151]:
nti_july_paris = nti_july_paris[
    ['idcar_200m', 'year', 'month',
     'early_morning', 'late_night',
     'NTI', 'geometry']
].copy()

In [152]:

nti_july_paris.to_file(
    'C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/ecostress_paris_july_heat_retention_200m.gpkg',
    driver='GPKG'
)

print("✅ Summer NTI saved as GeoPackage with geometry")

✅ Summer NTI saved as GeoPackage with geometry
